# Trabajo 2 Marketing 2026-1
## Starbucks America - Version 5 con StepMix sociodemografico

Esta version mantiene la segmentacion RFM con K-Means y cambia la segmentacion sociodemografica/conductual para que el modelo final sea **StepMix**. K-Means se conserva como comparacion metodologica.


---
## 0. Librerías

Instalamos StepMix si no esta disponible, porque lo usamos en la segmentacion sociodemografica.


In [1]:
# Instalar dependencias necesarias
import sys
!{sys.executable} -m pip install stepmix -q

Cargamos las librerias necesarias y dejamos listo el estilo de los graficos.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from stepmix import StepMix
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

SEED = 42
np.random.seed(SEED)

---
## 1. Carga y exploración inicial de datos

Cargamos la base original de ordenes y revisamos sus primeras filas.


In [3]:
df_raw = pd.read_csv('s_order.csv', parse_dates=['order_date'])
print(f'Filas: {df_raw.shape[0]:,}  |  Columnas: {df_raw.shape[1]}')
df_raw.head()

Filas: 100,000  |  Columnas: 20


,customer_id,order_id,order_date,order_time,day_of_week,order_channel,store_id,store_location_type,region,customer_age_group,customer_gender,is_rewards_member,cart_size,num_customizations,total_spend,fulfillment_time_min,drink_category,has_food_item,order_ahead,customer_satisfaction
0,CUST_12974,ORD_00000001,2024-03-25,08:47,Mon,Drive-Thru,STR_340,Suburban,Southwest,18-24,Male,False,5,0,14.48,8.2,Refresher,False,False,4
1,CUST_08235,ORD_00000002,2025-07-18,08:02,Fri,Mobile App,STR_425,Urban,Northeast,35-44,Female,True,1,3,9.52,5.4,Brewed Coffee,False,True,4
2,CUST_00393,ORD_00000003,2025-01-15,05:40,Wed,Kiosk,STR_103,Suburban,Midwest,25-34,Female,False,2,1,9.32,4.9,Brewed Coffee,False,False,5
3,CUST_06936,ORD_00000004,2024-07-30,15:10,Tue,Drive-Thru,STR_318,Suburban,Midwest,25-34,Female,True,2,1,9.55,3.5,Refresher,False,False,4
4,CUST_09800,ORD_00000005,2024-06-18,07:38,Tue,Drive-Thru,STR_338,Suburban,Northeast,35-44,Female,False,3,1,12.24,4.1,Frappuccino,False,False,3


Revisamos tipos de datos, valores faltantes y cantidad de valores distintos por columna.


In [4]:
# Tipos de datos y valores nulos
info = pd.DataFrame({
    'dtype': df_raw.dtypes,
    'nulos': df_raw.isnull().sum(),
    'pct_nulos': (df_raw.isnull().mean() * 100).round(2),
    'n_unicos': df_raw.nunique()
})
info

,dtype,nulos,pct_nulos,n_unicos
customer_id,object,0,0.0,14988
order_id,object,0,0.0,100000
order_date,datetime64[ns],0,0.0,730
order_time,object,0,0.0,1440
day_of_week,object,0,0.0,7
order_channel,object,0,0.0,4
store_id,object,0,0.0,500
store_location_type,object,0,0.0,3
region,object,0,0.0,5
customer_age_group,object,0,0.0,5


Miramos un resumen numerico para entender promedios, rangos y posibles valores extremos.


In [5]:
# Estadísticas descriptivas – variables numéricas
df_raw[['cart_size', 'num_customizations', 'total_spend',
        'fulfillment_time_min', 'customer_satisfaction']].describe().round(2)

,cart_size,num_customizations,total_spend,fulfillment_time_min,customer_satisfaction
count,100000.00,100000.00,100000.00,100000.00,100000.00
mean,3.74,1.81,14.87,4.55,3.69
std,1.70,1.46,5.51,1.55,1.18
min,1.00,0.00,3.51,1.00,1.00
25%,3.00,1.00,10.84,3.40,3.00
50%,4.00,2.00,14.17,4.40,4.00
75%,5.00,3.00,18.18,5.50,5.00
max,10.00,8.00,40.31,11.20,5.00


Revisamos como se distribuyen las principales variables categoricas de la base.


In [6]:
# Distribución de variables categóricas clave
cats = ['order_channel', 'store_location_type', 'region',
        'customer_age_group', 'customer_gender', 'drink_category']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(cats):
    counts = df_raw[col].value_counts()
    axes[i].bar(counts.index, counts.values,
                color=sns.color_palette('muted', len(counts)), edgecolor='white')
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].tick_params(axis='x', rotation=25)
    for j, v in enumerate(counts.values):
        axes[i].text(j, v + 100, f'{v/len(df_raw)*100:.1f}%', ha='center', fontsize=8)
plt.suptitle('Distribución de variables categóricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_categoricas.png', bbox_inches='tight')
plt.show()

Graficamos variables numericas para observar su comportamiento general antes de segmentar.


In [7]:
# Distribución de variables numéricas
nums = ['cart_size', 'num_customizations', 'total_spend', 'fulfillment_time_min']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, col in enumerate(nums):
    axes[i].hist(df_raw[col], bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(col.replace('_', ' ').title())
plt.suptitle('Distribución de variables numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_numericas.png', bbox_inches='tight')
plt.show()

---
## 2. Limpieza y preparación de datos

Creamos una copia de trabajo y hacemos ajustes basicos de limpieza.


In [8]:
df = df_raw.copy()

# ── 2.1  Verificar duplicados ─────────────────────────────────────────────
n_dup = df.duplicated().sum()
print(f'Filas duplicadas: {n_dup}')
df = df.drop_duplicates()

# ── 2.2  Tratar outliers en total_spend (IQR) ────────────────────────────
Q1, Q3 = df['total_spend'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 3 * IQR, Q3 + 3 * IQR
n_out = ((df['total_spend'] < lower) | (df['total_spend'] > upper)).sum()
print(f'Outliers en total_spend (3×IQR): {n_out} → reemplazados por límites')
df['total_spend'] = df['total_spend'].clip(lower, upper)

# ── 2.3  Extraer hora del pedido ─────────────────────────────────────────
df['order_hour'] = pd.to_datetime(df['order_time'], format='%H:%M').dt.hour
df['time_of_day'] = pd.cut(
    df['order_hour'],
    bins=[-1, 11, 16, 20, 24],
    labels=['Mañana', 'Tarde', 'Noche', 'Madrugada']
)

# ── 2.4  Codificar booleanos como 0/1 ───────────────────────────────────
bool_cols = ['is_rewards_member', 'has_food_item', 'order_ahead']
df[bool_cols] = df[bool_cols].astype(int)

print(f'\nDatos limpios: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head(3)

Filas duplicadas: 0
Outliers en total_spend (3×IQR): 1 → reemplazados por límites

Datos limpios: 100,000 filas × 22 columnas


,customer_id,order_id,order_date,order_time,day_of_week,order_channel,store_id,store_location_type,region,customer_age_group,...,cart_size,num_customizations,total_spend,fulfillment_time_min,drink_category,has_food_item,order_ahead,customer_satisfaction,order_hour,time_of_day
0,CUST_12974,ORD_00000001,2024-03-25,08:47,Mon,Drive-Thru,STR_340,Suburban,Southwest,18-24,...,5,0,14.48,8.2,Refresher,0,0,4,8,Mañana
1,CUST_08235,ORD_00000002,2025-07-18,08:02,Fri,Mobile App,STR_425,Urban,Northeast,35-44,...,1,3,9.52,5.4,Brewed Coffee,0,1,4,8,Mañana
2,CUST_00393,ORD_00000003,2025-01-15,05:40,Wed,Kiosk,STR_103,Suburban,Midwest,25-34,...,2,1,9.32,4.9,Brewed Coffee,0,0,5,5,Mañana


---
## 3. Construcción de métricas por cliente

Cada fila es una transacción. Se agregan por`customer_id`

Pasamos de una base por orden a una base por cliente, que es la unidad de nuestro analisis.


In [9]:
# Fecha de referencia: día siguiente al último pedido en la data
ref_date = df['order_date'].max() + pd.Timedelta(days=1)
print(f'Fecha de referencia para Recency: {ref_date.date()}')

cust = df.groupby('customer_id').agg(
    Recency      = ('order_date', lambda x: (ref_date - x.max()).days),
    Frequency    = ('order_id',   'count'),
    Monetary     = ('total_spend','sum'),
    avg_spend    = ('total_spend','mean'),
    avg_cart     = ('cart_size',  'mean'),
    avg_custom   = ('num_customizations', 'mean'),
    avg_satisf   = ('customer_satisfaction', 'mean'),
    pct_rewards  = ('is_rewards_member',    'mean'),
    pct_food     = ('has_food_item',        'mean'),
    pct_ahead    = ('order_ahead',          'mean'),
    top_channel  = ('order_channel',        lambda x: x.mode()[0]),
    top_region   = ('region',               lambda x: x.mode()[0]),
    top_location = ('store_location_type',  lambda x: x.mode()[0]),
    top_age      = ('customer_age_group',   lambda x: x.mode()[0]),
    top_gender   = ('customer_gender',      lambda x: x.mode()[0]),
    top_drink    = ('drink_category',       lambda x: x.mode()[0]),
    avg_fulfillment = ('fulfillment_time_min', 'mean'),
).reset_index()

print(f'Clientes únicos: {len(cust):,}')
cust[['customer_id','Recency','Frequency','Monetary','avg_spend','avg_cart','avg_satisf']].describe().round(2)

Fecha de referencia para Recency: 2025-12-31
Clientes únicos: 14,988


,Recency,Frequency,Monetary,avg_spend,avg_cart,avg_satisf
count,14988.00,14988.00,14988.00,14988.00,14988.00,14988.00
mean,108.59,6.67,99.19,14.87,3.74,3.69
std,105.03,2.56,40.97,2.47,0.74,0.51
min,1.00,1.00,3.83,3.83,1.00,1.00
25%,32.00,5.00,69.79,13.23,3.25,3.38
50%,76.00,6.00,95.60,14.76,3.71,3.71
75%,153.00,8.00,124.86,16.38,4.20,4.00
max,728.00,18.00,301.68,29.18,9.00,5.00


---
## 4. Segmentación RFM


### 4.1 Preparación de variables RFM

Preparamos las variables RFM y las dejamos en una escala comparable.


In [10]:
rfm_features = ['Recency', 'Frequency', 'Monetary']

X_rfm = cust[rfm_features].copy()

# K-Means requiere estandarización
scaler_rfm = StandardScaler()
X_rfm_sc = scaler_rfm.fit_transform(X_rfm)

print(f'\nShape matriz estandarizada: {X_rfm_sc.shape}')


Shape matriz estandarizada: (14988, 3)


### 4.2 K-Means RFM – Selección de k (Codo + Silhouette)

Probamos distintos valores de k para decidir una cantidad razonable de segmentos RFM.


In [11]:
K_range = range(2, 9)
inertias_rfm, sil_rfm, db_rfm, ch_rfm = [], [], [], []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labs = km.fit_predict(X_rfm_sc)
    inertias_rfm.append(km.inertia_)
    sil_rfm.append(silhouette_score(X_rfm_sc, labs))
    db_rfm.append(davies_bouldin_score(X_rfm_sc, labs))
    ch_rfm.append(calinski_harabasz_score(X_rfm_sc, labs))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ks = list(K_range)

axes[0].plot(ks, inertias_rfm, 'o-', color='steelblue', linewidth=2, markersize=7)
axes[0].set_title('Método del Codo (Inercia)')
axes[0].set_xlabel('Número de clusters'); axes[0].set_ylabel('Inercia')

axes[1].plot(ks, sil_rfm, 'o-', color='darkorange', linewidth=2, markersize=7)
axes[1].set_title('Coeficiente de Silhouette (↑ mejor)')
axes[1].set_xlabel('Número de clusters')

axes[2].plot(ks, db_rfm, 'o-', color='seagreen', linewidth=2, markersize=7)
axes[2].set_title('Davies-Bouldin (↓ mejor)')
axes[2].set_xlabel('Número de clusters')

for ax in axes:
    ax.grid(True, alpha=0.4)
plt.suptitle('Selección de k – RFM (K-Means)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_rfm_kmeans_k.png', bbox_inches='tight')
plt.show()

metricas_rfm_km = pd.DataFrame({'k': ks, 'Inercia': inertias_rfm,
                                  'Silhouette': sil_rfm, 'Davies-Bouldin': db_rfm,
                                  'Calinski-Harabasz': ch_rfm}).set_index('k')
print(f"Mejor k → Silhouette: {ks[int(np.argmax(sil_rfm))]}"
      f"  |  Davies-Bouldin: {ks[int(np.argmin(db_rfm))]}"
      f"  |  Calinski-Harabasz: {ks[int(np.argmax(ch_rfm))]}")
metricas_rfm_km.round(4)

  File "c:\Users\Dani\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\Dani\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dani\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dani\anaconda3\Lib\subprocess.py",

Mejor k → Silhouette: 2  |  Davies-Bouldin: 3  |  Calinski-Harabasz: 3


,Inercia,Silhouette,Davies-Bouldin,Calinski-Harabasz
k,,,,
2,24866.3107,0.3871,0.9538,12112.1370
3,16807.0856,0.3868,0.8874,12552.2234
4,12881.0684,0.3387,0.9246,12440.3270
5,10739.5167,0.3483,0.9015,11937.0806
6,9089.7736,0.3225,0.8994,11825.8423
7,7988.9576,0.3157,0.8957,11556.1989
8,7185.3855,0.2963,0.9433,11251.7929


### 4.3 StepMix (Latent Class) RFM – Selección de k (BIC + Entropía)

Usamos StepMix en RFM solo como comparacion frente a K-Means.


In [12]:
# StepMix: BIC + entropia de clasificacion
# La entropia se calcula directamente desde las probabilidades posteriores del modelo.
# Se evita el uso de umbrales arbitrarios o ajustes artificiales, para no alterar la lectura de la entropia.
def stepmix_entropy(model, X):
    probs = model.predict_proba(X)
    probs = np.asarray(probs, dtype=float)
    probs = np.clip(probs, 1e-12, 1.0)
    probs = probs / probs.sum(axis=1, keepdims=True)
    H = -np.sum(probs * np.log(probs), axis=1).mean()
    return 1 - H / np.log(model.n_components)

bic_rfm, ent_class_rfm = [], []
for k in K_range:
    sm = StepMix(n_components=k, measurement='continuous', random_state=SEED, verbose=0)
    sm.fit(X_rfm)
    bic_rfm.append(sm.bic(X_rfm))
    ent_class_rfm.append(stepmix_entropy(sm, X_rfm))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ks = list(K_range)
axes[0].plot(ks, bic_rfm, 's-', color='steelblue', linewidth=2, markersize=7)
axes[0].set_title('BIC (menor es mejor)')
axes[0].set_xlabel('Numero de componentes'); axes[0].set_ylabel('BIC')
axes[0].grid(True, alpha=0.4)
axes[1].plot(ks, ent_class_rfm, 's-', color='darkorange', linewidth=2, markersize=7)
axes[1].axhline(1.0, color='red', linestyle='--', alpha=0.5, label='Sep. perfecta (=1, revisar)')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Entropia de clasificacion (cercano a 1 = mayor certeza)')
axes[1].set_xlabel('Numero de componentes'); axes[1].set_ylabel('Entropia')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.4)
plt.suptitle('Seleccion de k - RFM (StepMix / Latent Class)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_rfm_stepmix_k.png', bbox_inches='tight')
plt.show()
metricas_rfm_sm = pd.DataFrame({'k': ks, 'BIC': bic_rfm,
                                'Entropia clasificacion': ent_class_rfm}).set_index('k')
print(f'Mejor k por BIC: {ks[int(np.argmin(bic_rfm))]}  |  '
      f'Mejor k por Entropia: {ks[int(np.argmax(ent_class_rfm))]}')
metricas_rfm_sm.round(4)


Fitting StepMix...


Initializations (n_init) :   0%|          | 0/1 [00:00<?, ?it/s]

Initializations (n_init) : 100%|██████████| 1/1 [00:00<00:00,  1.91it/s, max_LL=-1.95e+5, max_avg_LL=-13]


Fitting StepMix...


Initializations (n_init) : 100%|██████████| 1/1 [00:01<00:00,  1.48s/it, max_LL=-1.91e+5, max_avg_LL=-12.7]


...

Initializations (n_init) : 100%|██████████| 1/1 [00:05<00:00,  5.01s/it, max_LL=-1.85e+5, max_avg_LL=-12.3]


Fitting StepMix...


Initializations (n_init) : 100%|██████████| 1/1 [00:05<00:00,  5.48s/it, max_LL=-1.84e+5, max_avg_LL=-12.3]


Mejor k por BIC: 8  |  Mejor k por Entropia: 5


,BIC,Entropia clasificacion
k,,
2,390431.6482,0.7545
3,382067.6330,0.8149
4,377007.6481,0.8361
5,373978.6793,0.8433
6,372190.6628,0.8087
7,370627.3832,0.8139
8,369274.1005,0.7926


### 4.4 Selección del método y k óptimo RFM

Entrenamos el modelo RFM final, asignamos segmentos y agregamos los nombres definidos por el grupo.


In [13]:
K_RFM = 3  

# Modelo elegido: K-Means 
km_rfm = KMeans(n_clusters=K_RFM, random_state=SEED, n_init=20)
cust['seg_rfm'] = km_rfm.fit_predict(X_rfm_sc)

# Nombres RFM definidos por el grupo.
# Se crean temprano para que todos los graficos muestren nombres y no solo codigos.
nombres_rfm = {
    0: 'Cliente Espontaneo',
    1: 'Cliente Estrella',
    2: 'Cliente Potencial',
}
cust['seg_rfm_nombre'] = cust['seg_rfm'].map(nombres_rfm)
orden_rfm_nombres = [nombres_rfm[i] for i in sorted(nombres_rfm)]

sil_final_rfm = silhouette_score(X_rfm_sc, cust['seg_rfm'])
db_final_rfm  = davies_bouldin_score(X_rfm_sc, cust['seg_rfm'])

print(f'Metodo elegido: K-Means  |  k = {K_RFM}')
print(f'Silhouette: {sil_final_rfm:.4f}  |  Davies-Bouldin: {db_final_rfm:.4f}')
print('\nTamano de segmentos:')
print(cust['seg_rfm_nombre'].value_counts().reindex(orden_rfm_nombres))


Metodo elegido: K-Means  |  k = 3
Silhouette: 0.3868  |  Davies-Bouldin: 0.8874

Tamano de segmentos:
seg_rfm_nombre
Cliente Espontaneo    2344
Cliente Estrella      5254
Cliente Potencial     7390
Name: count, dtype: int64


### 4.5 Caracterización de segmentos RFM

Construimos una tabla resumen para interpretar el perfil promedio de cada segmento RFM.


In [14]:
# Perfil base R, F, M + variables auxiliares para interpretación
aux_vars = ['avg_spend', 'avg_satisf', 'pct_rewards', 'pct_food', 'pct_ahead']
rfm_profile = cust.groupby('seg_rfm')[rfm_features + aux_vars].mean().round(2)
rfm_profile['n_clientes'] = cust.groupby('seg_rfm').size()
rfm_profile['pct_mercado'] = (rfm_profile['n_clientes'] / len(cust) * 100).round(1)
rfm_profile

,Recency,Frequency,Monetary,avg_spend,avg_satisf,pct_rewards,pct_food,pct_ahead,n_clientes,pct_mercado
seg_rfm,,,,,,,,,,
0,296.92,4.18,61.22,14.74,3.69,0.48,0.30,0.29,2344,15.6
1,70.37,9.40,142.75,15.27,3.69,0.48,0.32,0.32,5254,35.1
2,76.03,5.52,80.27,14.63,3.69,0.47,0.31,0.29,7390,49.3


Comparamos visualmente recencia, frecuencia y gasto mediante un grafico radar.


In [15]:
# Radar chart por segmento RFM
radar_vars = rfm_features.copy()   # R, F, M
rfm_norm = rfm_profile[radar_vars].copy()

# Recency se invierte porque menos dias desde la ultima compra significa cliente mas reciente.
rfm_norm['Recency'] = 1 - (rfm_norm['Recency'] - rfm_norm['Recency'].min()) /                           (rfm_norm['Recency'].max() - rfm_norm['Recency'].min() + 1e-9)
for col in ['Frequency', 'Monetary']:
    rfm_norm[col] = (rfm_norm[col] - rfm_norm[col].min()) /                     (rfm_norm[col].max() - rfm_norm[col].min() + 1e-9)

labels_radar = ['Recency\n(inv.)', 'Frequency', 'Monetary']
N = len(labels_radar)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, axes = plt.subplots(1, K_RFM, figsize=(4.5 * K_RFM, 4), subplot_kw=dict(polar=True))
colors = sns.color_palette('tab10', K_RFM)
if K_RFM == 1:
    axes = [axes]

for seg in range(K_RFM):
    vals = rfm_norm.loc[seg, radar_vars].tolist() + [rfm_norm.loc[seg, radar_vars[0]]]
    ax = axes[seg]
    ax.plot(angles, vals, color=colors[seg], linewidth=2)
    ax.fill(angles, vals, color=colors[seg], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels_radar, size=9)
    ax.set_yticklabels([])
    nombre = nombres_rfm.get(seg, f'Seg. RFM {seg}')
    ax.set_title(f'{nombre}\n(n={rfm_profile.loc[seg,"n_clientes"]:,})',
                 fontsize=10, pad=15)

plt.suptitle('Perfiles RFM - valores normalizados', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_rfm_radar.png', bbox_inches='tight')
plt.show()


Revisamos la dispersion interna de cada segmento RFM con boxplots.


In [16]:
# Boxplots R, F, M por segmento RFM con nombres
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = sns.color_palette('tab10', K_RFM)
for i, var in enumerate(rfm_features):
    sns.boxplot(
        data=cust,
        x='seg_rfm_nombre',
        y=var,
        order=orden_rfm_nombres,
        palette='tab10',
        ax=axes[i],
        flierprops=dict(marker='o', alpha=0.3, markersize=3),
    )
    axes[i].set_title(f'{var} por segmento RFM')
    axes[i].set_xlabel('Segmento RFM')
    axes[i].tick_params(axis='x', rotation=15)
plt.suptitle('Distribucion de R, F, M por segmento', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_rfm_boxplot.png', bbox_inches='tight')
plt.show()


Reducimos las variables RFM a tres dimensiones para visualizar los segmentos en 3D.


In [17]:
# Visualizacion 3D mejorada - Segmentacion RFM (muestra para claridad visual)
pca_rfm = PCA(n_components=3, random_state=SEED)
X_pca_rfm = pca_rfm.fit_transform(X_rfm_sc)
centroids_rfm_pca = pca_rfm.transform(km_rfm.cluster_centers_)

# Muestra aleatoria para no saturar el grafico (solo visual, no afecta el modelo)
sample_idx = np.random.choice(len(X_pca_rfm), min(5000, len(X_pca_rfm)), replace=False)

colors_tab = plt.cm.tab10.colors

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#ffffff')

for seg in range(K_RFM):
    mask = (cust['seg_rfm'].iloc[sample_idx].values == seg)
    nombre = nombres_rfm.get(seg, f'Seg. RFM {seg}')
    ax.scatter(
        X_pca_rfm[sample_idx][mask, 0],
        X_pca_rfm[sample_idx][mask, 1],
        X_pca_rfm[sample_idx][mask, 2],
        c=[colors_tab[seg % 10]],
        alpha=0.15, s=5, depthshade=False,
        label=nombre)

ax.scatter(
    centroids_rfm_pca[:, 0], centroids_rfm_pca[:, 1], centroids_rfm_pca[:, 2],
    c='black', s=900, marker='X', edgecolors='yellow', linewidths=3,
    depthshade=False, zorder=10, label='Centroides')

for seg in range(K_RFM):
    nombre_corto = nombres_rfm.get(seg, f'S{seg}').replace('Cliente ', '')
    ax.text(centroids_rfm_pca[seg, 0], centroids_rfm_pca[seg, 1],
            centroids_rfm_pca[seg, 2] + 0.15,
            nombre_corto, fontsize=10, fontweight='bold', color='black',
            ha='center', zorder=11)

var_exp = pca_rfm.explained_variance_ratio_
ax.set_xlabel(f'PC1 ({var_exp[0]*100:.1f}% var)', labelpad=8)
ax.set_ylabel(f'PC2 ({var_exp[1]*100:.1f}% var)', labelpad=8)
ax.set_zlabel(f'PC3 ({var_exp[2]*100:.1f}% var)', labelpad=8)
ax.set_title('Segmentos RFM - Espacio PCA 3D', fontsize=13, fontweight='bold', pad=12)
ax.legend(loc='upper left', fontsize=9, framealpha=0.8)
ax.view_init(elev=20, azim=45)

plt.tight_layout()
plt.savefig('fig_rfm_pca3d.png', bbox_inches='tight', dpi=130)
plt.show()
print(f'Varianza explicada acumulada: {var_exp.sum()*100:.1f}%')


Varianza explicada acumulada: 100.0%


---
## 5. Segmentacion Sociodemografica 


### 5.1 Seleccion de variables y preparacion para K-Means y StepMix

Se preparan dos matrices distintas:

- **K-Means:** usa variables numericas y variables categoricas transformadas a columnas dummy. Se mantiene solo como punto de comparacion.
- **StepMix:** usa variables numericas estandarizadas y variables categoricas codificadas como categorias. Esta sera la base del modelo final sociodemografico/conductual.


Preparamos las variables de perfil y comportamiento para comparar K-Means y StepMix.


In [18]:
# Variables sociodemograficas y conductuales del cliente
# Las numericas describen comportamiento; las categoricas describen perfil y contexto de compra.
socio_cats = ['top_age', 'top_gender', 'top_region', 'top_location', 'top_channel']
socio_nums = ['avg_spend', 'avg_satisf', 'pct_rewards', 'pct_food', 'pct_ahead']

# -------------------------------------------------------------------
# Matriz para K-Means: dummies + escalado.
# Se usa para comparar, no como modelo final de esta version.
# -------------------------------------------------------------------
socio_ohe = pd.get_dummies(cust[socio_cats], drop_first=False)
X_socio_km = pd.concat([
    cust[socio_nums].reset_index(drop=True),
    socio_ohe.reset_index(drop=True)
], axis=1)

scaler_soc = StandardScaler()
X_socio_sc = scaler_soc.fit_transform(X_socio_km)

print(f'Dimensiones K-Means (OHE + escalado): {X_socio_sc.shape}')
print(f'  Numericas: {len(socio_nums)} | Categoricas OHE: {socio_ohe.shape[1]}')

# -------------------------------------------------------------------
# Matriz para StepMix mixto:
# - numericas: estandarizadas para que queden en escala comparable;
# - categoricas: codificadas como enteros, sin convertirlas en muchas dummies.
# -------------------------------------------------------------------
from sklearn.preprocessing import OrdinalEncoder

scaler_soc_sm = StandardScaler()
X_socio_sm_num = scaler_soc_sm.fit_transform(cust[socio_nums])

enc_soc = OrdinalEncoder()
X_socio_sm_cat = enc_soc.fit_transform(cust[socio_cats]).astype(int)

X_socio_sm = np.hstack([X_socio_sm_num, X_socio_sm_cat])

# Descriptor del modelo mixto de StepMix.
# Primero van las 5 variables numericas como continuas.
# Luego cada variable categorica se declara como multinoulli.
measurement_soc = {
    'num': {'model': 'continuous', 'n_columns': len(socio_nums)}
}
for col, cats in zip(socio_cats, enc_soc.categories_):
    measurement_soc[f'cat_{col}'] = {
        'model': 'multinoulli',
        'n_columns': 1,
        'integer_codes': True,
        'max_n_outcomes': len(cats),
    }

print(f'Dimensiones StepMix mixto: {X_socio_sm.shape}')
print('Variables categoricas usadas por StepMix:')
for col, cats in zip(socio_cats, enc_soc.categories_):
    print(f'  {col}: {list(cats)}')


Dimensiones K-Means (OHE + escalado): (14988, 26)
  Numericas: 5 | Categoricas OHE: 21
Dimensiones StepMix mixto: (14988, 10)
Variables categoricas usadas por StepMix:
  top_age: ['18-24', '25-34', '35-44', '45-54', '55+']
  top_gender: ['Female', 'Male', 'Non-binary', 'Prefer not to say']
  top_region: ['Midwest', 'Northeast', 'Southeast', 'Southwest', 'West']
  top_location: ['Rural', 'Suburban', 'Urban']
  top_channel: ['Drive-Thru', 'In-Store Cashier', 'Kiosk', 'Mobile App']


### 5.2 K-Means Sociodemográfico – Selección de k

Probamos K-Means sociodemografico como referencia para comparar resultados.


In [19]:
sil_soc_km, db_soc_km, inert_soc_km = [], [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labs = km.fit_predict(X_socio_sc)
    sil_soc_km.append(silhouette_score(X_socio_sc, labs))
    db_soc_km.append(davies_bouldin_score(X_socio_sc, labs))
    inert_soc_km.append(km.inertia_)

ks = list(K_range)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(ks, inert_soc_km, 'o-', color='steelblue', linewidth=2, markersize=7)
axes[0].set_title('Método del Codo (Inercia)'); axes[0].set_xlabel('k')
axes[1].plot(ks, sil_soc_km, 'o-', color='darkorange', linewidth=2, markersize=7)
axes[1].set_title('Silhouette (↑ mejor)'); axes[1].set_xlabel('k')
axes[2].plot(ks, db_soc_km, 'o-', color='seagreen', linewidth=2, markersize=7)
axes[2].set_title('Davies-Bouldin (↓ mejor)'); axes[2].set_xlabel('k')
for ax in axes:
    ax.grid(True, alpha=0.4)
plt.suptitle('Selección de k – Sociodemográfico (K-Means)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_soc_kmeans_k.png', bbox_inches='tight')
plt.show()

metricas_soc_km = pd.DataFrame({'k': ks, 'Inercia': inert_soc_km,
                                  'Silhouette': sil_soc_km, 'Davies-Bouldin': db_soc_km}).set_index('k')
print(f"Mejor k → Silhouette: {ks[int(np.argmax(sil_soc_km))]}"
      f"  |  Davies-Bouldin: {ks[int(np.argmin(db_soc_km))]}")
metricas_soc_km.round(4)

Mejor k → Silhouette: 6  |  Davies-Bouldin: 6


,Inercia,Silhouette,Davies-Bouldin
k,,,
2,350253.1325,0.1018,2.9185
3,330173.5967,0.1188,2.7510
4,319320.4929,0.0823,2.9830
5,303479.6186,0.1252,2.4696
6,291283.8736,0.1304,2.3025
7,283028.2997,0.1113,2.4051
8,274976.4189,0.1082,2.4747


### 5.3 StepMix Sociodemografico - Seleccion de k (BIC + Entropia)

StepMix se evalua con dos criterios principales:

- **BIC:** mientras menor, mejor ajuste penalizado por complejidad.
- **Entropia de clasificacion:** mientras mas alta, mas clara es la separacion entre clases. Una entropia alta ayuda, pero no reemplaza la interpretacion de marketing.

En esta version, StepMix es el metodo elegido para la segmentacion sociodemografica/conductual.


Probamos StepMix con variables numericas y categoricas para elegir el numero de clases.


In [20]:
# StepMix sociodemografico/conductual con modelo mixto.
# Este modelo usa numericas como continuas y categoricas como multinoulli.
bic_soc, ent_class_soc = [], []
sm_soc_models = {}
K_range = range(2, 9)

SM_N_INIT = 3
SM_MAX_ITER = 300

for k in K_range:
    sm = StepMix(
        n_components=k,
        measurement=measurement_soc,
        random_state=SEED,
        n_init=SM_N_INIT,
        max_iter=SM_MAX_ITER,
        init_params='kmeans',
        verbose=0,
        progress_bar=0,
    )
    sm.fit(X_socio_sm)
    sm_soc_models[k] = sm
    bic_soc.append(sm.bic(X_socio_sm))
    ent_class_soc.append(stepmix_entropy(sm, X_socio_sm))

ks = list(K_range)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks, bic_soc, 's-', color='steelblue', linewidth=2, markersize=7)
axes[0].set_title('BIC (menor es mejor)')
axes[0].set_xlabel('Numero de clases'); axes[0].set_ylabel('BIC')
axes[0].grid(True, alpha=0.4)

axes[1].plot(ks, ent_class_soc, 's-', color='darkorange', linewidth=2, markersize=7)
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Entropia de clasificacion (mayor es mejor)')
axes[1].set_xlabel('Numero de clases'); axes[1].set_ylabel('Entropia')
axes[1].grid(True, alpha=0.4)

plt.suptitle('Seleccion de k - Sociodemografico (StepMix mixto)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_soc_stepmix_k_v5.png', bbox_inches='tight')
plt.show()

metricas_soc_sm = pd.DataFrame({
    'k': ks,
    'BIC': bic_soc,
    'Entropia clasificacion': ent_class_soc,
}).set_index('k')

print(f'Mejor k por BIC: {ks[int(np.argmin(bic_soc))]} | '
      f'Mejor k por entropia: {ks[int(np.argmax(ent_class_soc))]}')
metricas_soc_sm.round(4)


Mejor k por BIC: 8 | Mejor k por entropia: 4


,BIC,Entropia clasificacion
k,,
2,384782.5459,0.8504
3,384967.4885,0.9056
4,385184.0288,0.9255
5,352283.5988,0.7977
6,351275.0197,0.7581
7,351479.0291,0.7924
8,351272.5783,0.7408


### 5.4 Seleccion del metodo y k final Sociodemografico

El modelo final sociodemografico/conductual se estima con **StepMix mixto**, ya que este metodo permite trabajar de mejor manera con una combinacion de variables numericas y categoricas. Esta decision se justifica porque, al usar K-Means en la segmentacion sociodemografica, los grupos quedaban muy similares entre si y se diferenciaban principalmente por pocas variables, como edad y canal.

Se selecciona **k = 5** porque entrega un equilibrio adecuado entre ajuste estadistico, claridad de clasificacion e interpretabilidad comercial. Aunque el BIC y la entropia se usan como referencia, la decision final tambien considera que los cinco perfiles resultantes son accionables para marketing: diferencian edad, canal principal, nivel de digitalizacion, tipo de localidad y potencial de consumo.


Entrenamos el modelo StepMix final y asignamos cada cliente a un segmento sociodemografico.


In [21]:
K_SOC = 5

# Modelo elegido: StepMix mixto con k=5.
# k=5 se elige por equilibrio entre BIC, entropia e interpretabilidad comercial de los perfiles.
# Modelo elegido en v5: StepMix mixto
# Se usa la asignacion modal: cada cliente queda en la clase con mayor probabilidad posterior.
sm_soc_final = sm_soc_models.get(K_SOC)
if sm_soc_final is None:
    sm_soc_final = StepMix(
        n_components=K_SOC,
        measurement=measurement_soc,
        random_state=SEED,
        n_init=SM_N_INIT,
        max_iter=SM_MAX_ITER,
        init_params='kmeans',
        verbose=0,
        progress_bar=0,
    )
    sm_soc_final.fit(X_socio_sm)

cust['seg_soc'] = sm_soc_final.predict(X_socio_sm)
soc_proba = sm_soc_final.predict_proba(X_socio_sm)
cust['seg_soc_prob_max'] = soc_proba.max(axis=1)

bic_final_soc = sm_soc_final.bic(X_socio_sm)
ent_final_soc = stepmix_entropy(sm_soc_final, X_socio_sm)

print(f'Metodo elegido: StepMix mixto | k = {K_SOC}')
print('Justificacion: k=5 entrega segmentos interpretables y accionables, manteniendo buen ajuste y claridad de clasificacion.')
print(f'BIC: {bic_final_soc:.1f} | Entropia: {ent_final_soc:.4f}')
print(f'Probabilidad posterior promedio de asignacion: {cust["seg_soc_prob_max"].mean():.4f}')
print('\nTamano de segmentos:')
print(cust['seg_soc'].value_counts().sort_index())


Metodo elegido: StepMix mixto | k = 5
Justificacion: k=5 entrega segmentos interpretables y accionables, manteniendo buen ajuste y claridad de clasificacion.
BIC: 352283.6 | Entropia: 0.7977
Probabilidad posterior promedio de asignacion: 0.8532

Tamano de segmentos:
seg_soc
0    2302
1    2740
2    2287
3    2633
4    5026
Name: count, dtype: int64


### 5.5 Caracterización de segmentos sociodemográficos

Calculamos promedios numericos para describir cada segmento sociodemografico.


In [22]:
# Perfil numérico
soc_profile_num = cust.groupby('seg_soc')[socio_nums].mean().round(2)
soc_profile_num['n_clientes'] = cust.groupby('seg_soc').size()
soc_profile_num['pct_mercado'] = (soc_profile_num['n_clientes'] / len(cust) * 100).round(1)
soc_profile_num

,avg_spend,avg_satisf,pct_rewards,pct_food,pct_ahead,n_clientes,pct_mercado
seg_soc,,,,,,,
0,14.52,3.63,0.47,0.31,0.32,2302,15.4
1,13.60,3.63,0.41,0.27,0.17,2740,18.3
2,14.78,3.68,0.47,0.30,0.32,2287,15.3
3,13.24,3.61,0.39,0.27,0.00,2633,17.6
4,16.62,3.79,0.57,0.37,0.51,5026,33.5


Identificamos la categoria mas frecuente de cada variable dentro de cada segmento.


In [23]:
# Moda de variables categóricas por segmento
soc_profile_cat = cust.groupby('seg_soc')[socio_cats].agg(lambda x: x.mode()[0])
soc_profile_cat

,top_age,top_gender,top_region,top_location,top_channel
seg_soc,,,,,
0,35-44,Female,Midwest,Suburban,Mobile App
1,35-44,Male,Midwest,Rural,Drive-Thru
2,35-44,Male,Southwest,Rural,Mobile App
3,55+,Male,Midwest,Rural,Drive-Thru
4,25-34,Female,Midwest,Rural,Mobile App


Asignamos nombres a los segmentos sociodemograficos y revisamos su distribucion.


In [24]:
# Nombres de segmentos sociodemograficos StepMix
# Los nombres comerciales se basan en el rasgo dominante de cada segmento:
# canal principal, edad, nivel de digitalizacion, tipo de localidad y potencial de consumo.
nombres_soc = {
    0: 'Suburban Pro',       # Adultos digitales suburbanos, uso estable de app
    1: 'Practical Drive-Thru',    # Clientes tradicionales y practicos
    2: 'D-Frontier',        # Rurales pero altamente digitales
    3: 'Classic n Quick',         # Seniors orientados a rapidez y habito
    4: 'Smart Coffee',            # Jovenes digitales, alto engagement y gasto
}

descripcion_soc = {
    0: 'Adultos digitales suburbanos, uso estable de app',
    1: 'Clientes tradicionales y practicos',
    2: 'Rurales pero altamente digitales',
    3: 'Seniors orientados a rapidez y habito',
    4: 'Jovenes digitales, alto engagement y gasto',
}

# Agregar columnas con nombres al dataframe principal
cust['seg_rfm_nombre'] = cust['seg_rfm'].map(nombres_rfm)
cust['seg_soc_nombre'] = cust['seg_soc'].map(nombres_soc)
cust['seg_soc_descripcion'] = cust['seg_soc'].map(descripcion_soc)

# Perfiles con nombres
soc_num_named = soc_profile_num.copy()
soc_num_named.index = soc_num_named.index.map(nombres_soc)
soc_cat_named = soc_profile_cat.copy()
soc_cat_named.index = soc_cat_named.index.map(nombres_soc)

print('Perfil numerico por segmento sociodemografico StepMix:')
display(soc_num_named.round(2))
print('\nPerfil categorico por segmento sociodemografico StepMix:')
display(soc_cat_named)
print('\nDescripcion comercial de segmentos sociodemograficos:')
display(pd.Series(descripcion_soc, name='descripcion').rename_axis('seg_soc').to_frame().assign(nombre=pd.Series(nombres_soc)).loc[:, ['nombre', 'descripcion']])
print('\nProbabilidad posterior promedio por segmento:')
display(cust.groupby('seg_soc')['seg_soc_prob_max'].mean().round(4).rename('probabilidad_promedio'))
print('\nDistribucion RFM:')
print(cust['seg_rfm_nombre'].value_counts())
print('\nDistribucion Sociodemografico StepMix:')
print(cust['seg_soc_nombre'].value_counts())


Perfil numerico por segmento sociodemografico StepMix:


,avg_spend,avg_satisf,pct_rewards,pct_food,pct_ahead,n_clientes,pct_mercado
seg_soc,,,,,,,
Suburban Pro,14.52,3.63,0.47,0.31,0.32,2302,15.4
Practical Drive-Thru,13.60,3.63,0.41,0.27,0.17,2740,18.3
D-Frontier,14.78,3.68,0.47,0.30,0.32,2287,15.3
Classic n Quick,13.24,3.61,0.39,0.27,0.00,2633,17.6
Smart Coffee,16.62,3.79,0.57,0.37,0.51,5026,33.5



Perfil categorico por segmento sociodemografico StepMix:


,top_age,top_gender,top_region,top_location,top_channel
seg_soc,,,,,
Suburban Pro,35-44,Female,Midwest,Suburban,Mobile App
Practical Drive-Thru,35-44,Male,Midwest,Rural,Drive-Thru
D-Frontier,35-44,Male,Southwest,Rural,Mobile App
Classic n Quick,55+,Male,Midwest,Rural,Drive-Thru
Smart Coffee,25-34,Female,Midwest,Rural,Mobile App



Descripcion comercial de segmentos sociodemograficos:


,nombre,descripcion
seg_soc,,
0,Suburban Pro,"Adultos digitales suburbanos, uso estable de app"
1,Practical Drive-Thru,Clientes tradicionales y practicos
2,D-Frontier,Rurales pero altamente digitales
3,Classic n Quick,Seniors orientados a rapidez y habito
4,Smart Coffee,"Jovenes digitales, alto engagement y gasto"



Probabilidad posterior promedio por segmento:


seg_soc
0    0.8165
1    0.8309
2    0.7334
3    0.9989
4    0.8602
Name: probabilidad_promedio, dtype: float64


Distribucion RFM:
seg_rfm_nombre
Cliente Potencial     7390
Cliente Estrella      5254
Cliente Espontaneo    2344
Name: count, dtype: int64

Distribucion Sociodemografico StepMix:
seg_soc_nombre
Smart Coffee            5026
Practical Drive-Thru    2740
Classic n Quick         2633
Suburban Pro            2302
D-Frontier              2287
Name: count, dtype: int64


Graficamos la composicion de los segmentos para entender mejor que caracteriza a cada grupo.


In [25]:
# Composición categórica por segmento (barras apiladas %)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for i, col in enumerate(socio_cats):
    ct = cust.groupby(['seg_soc', col]).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.index = ct_pct.index.map(nombres_soc)
    ct_pct.plot(kind='bar', ax=axes[i], colormap='tab10', edgecolor='white')
    axes[i].set_title(col.replace('top_', '').replace('_', ' ').title())
    axes[i].set_xlabel('Segmento Sociodemográfico'); axes[i].set_ylabel('%')
    axes[i].tick_params(axis='x', rotation=15)
    axes[i].legend(title=col.replace('top_',''), bbox_to_anchor=(1, 1), fontsize=7)
# Ocultar el sexto panel si solo hay 5 variables
if len(socio_cats) < 6:
    axes[5].set_visible(False)
plt.suptitle('Composición categórica por segmento sociodemográfico',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_soc_composicion.png', bbox_inches='tight')
plt.show()

---
## 6. Comparacion de modelos

Se comparan K-Means y StepMix para cada segmentacion. En esta v5 se mantiene RFM con K-Means, pero la segmentacion sociodemografica/conductual final usa StepMix mixto.


Comparamos los metodos utilizados y dejamos claro cual se eligio en cada segmentacion.


In [26]:
best_k_rfm_km = ks[int(np.argmax(sil_rfm))]
best_k_rfm_sm = ks[int(np.argmin(bic_rfm))]
best_k_soc_km = ks[int(np.argmax(sil_soc_km))]
best_k_soc_sm = ks[int(np.argmin(bic_soc))]

comparacion = pd.DataFrame({
    'Segmentacion': ['RFM', 'RFM', 'Sociodemografico', 'Sociodemografico'],
    'Metodo': ['K-Means', 'StepMix', 'K-Means', 'StepMix mixto'],
    'Mejor k segun metrica': [best_k_rfm_km, best_k_rfm_sm, best_k_soc_km, best_k_soc_sm],
    'k elegido final': [K_RFM, '-', '-', K_SOC],
    'Metrica principal': [
        round(max(sil_rfm), 4),
        round(min(bic_rfm), 1),
        round(max(sil_soc_km), 4),
        round(bic_final_soc, 1),
    ],
    'Seleccionado': ['Si', 'No', 'No', 'Si'],
    'Justificacion': [
        'RFM se mantiene con K-Means porque separa bien recencia, frecuencia y gasto',
        'StepMix RFM se usa solo como contraste metodologico',
        'K-Means socio queda como comparacion, no como modelo final en v5',
        'StepMix mixto permite modelar numericas y categoricas; k=5 se mantiene por equilibrio entre ajuste, entropia e interpretabilidad comercial',
    ]
})
comparacion.set_index(['Segmentacion', 'Metodo'])


Mejor k segun metrica k elegido final  \
Segmentacion     Metodo                                                 
RFM              K-Means                            2               3   
                 StepMix                            8               -   
Sociodemografico K-Means                            6               -   
                 StepMix mixto                      8               5   

                                Metrica principal Seleccionado  \
Segmentacion     Metodo                                          
RFM              K-Means                   0.3871           Si   
                 StepMix              369274.1000           No   
Sociodemografico K-Means                   0.1304           No   
                 StepMix mixto        352283.6000           Si   

                                                                    Justificacion  
Segmentacion     Metodo                                                            
RFM              K-Means        RFM se mantiene con K-Means porque separa bien...  
                 StepMix        StepMix RFM se usa solo como contraste metodol...  
Sociodemografico K-Means        K-Means socio queda como comparacion, no como ...  
                 StepMix mixto  StepMix mixto permite modelar numericas y cate...

---
## 7. Segmentos de mercado: Matriz RFM × Sociodemográfico

Al cruzar los dos clusters se obtienen **k₁ × k₂ segmentos de mercado** distintos.

Cruzamos los segmentos RFM con los segmentos StepMix para formar mercados potenciales.


In [27]:
# Cruce de segmentos con nombres
cruce = pd.crosstab(cust['seg_rfm_nombre'], cust['seg_soc_nombre'],
                    margins=True, margins_name='Total')
cruce_pct = pd.crosstab(cust['seg_rfm_nombre'], cust['seg_soc_nombre'],
                         normalize='all').mul(100).round(1)

# Ordenar filas RFM segun la logica definida por el grupo.
cruce = cruce.reindex(index=orden_rfm_nombres + ['Total'])
cruce_pct = cruce_pct.reindex(index=orden_rfm_nombres)

print('Cruce: RFM (filas) x Sociodemografico StepMix (columnas) - conteos')
display(cruce)
print('\nPorcentaje del mercado total:')
display(cruce_pct)


Cruce: RFM (filas) x Sociodemografico StepMix (columnas) - conteos


seg_soc_nombre,Classic n Quick,D-Frontier,Practical Drive-Thru,Smart Coffee,Suburban Pro,Total
seg_rfm_nombre,,,,,,
Cliente Espontaneo,725,290,257,688,384,2344
Cliente Estrella,349,910,1180,2062,753,5254
Cliente Potencial,1559,1087,1303,2276,1165,7390
Total,2633,2287,2740,5026,2302,14988



Porcentaje del mercado total:


seg_soc_nombre,Classic n Quick,D-Frontier,Practical Drive-Thru,Smart Coffee,Suburban Pro
seg_rfm_nombre,,,,,
Cliente Espontaneo,4.8,1.9,1.7,4.6,2.6
Cliente Estrella,2.3,6.1,7.9,13.8,5.0
Cliente Potencial,10.4,7.3,8.7,15.2,7.8


Mostramos el cruce RFM x StepMix en mapas de calor para detectar grupos relevantes.


In [28]:
# Heatmap de la matriz de segmentos con nombres
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

sns.heatmap(cruce.iloc[:-1, :-1], annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, ax=axes[0])
axes[0].set_title('Conteo de clientes por segmento\n(RFM x Sociodemografico StepMix)')
axes[0].set_xlabel('Segmento Sociodemografico StepMix')
axes[0].set_ylabel('Segmento RFM')
axes[0].tick_params(axis='x', rotation=30)
axes[0].tick_params(axis='y', rotation=0)

sns.heatmap(cruce_pct, annot=True, fmt='.1f', cmap='OrRd',
            linewidths=0.5, ax=axes[1])
axes[1].set_title('% del mercado total\n(RFM x Sociodemografico StepMix)')
axes[1].set_xlabel('Segmento Sociodemografico StepMix')
axes[1].set_ylabel('Segmento RFM')
axes[1].tick_params(axis='x', rotation=30)
axes[1].tick_params(axis='y', rotation=0)

plt.suptitle(f'Matriz de segmentos de mercado ({K_RFM}x{K_SOC} = {K_RFM*K_SOC} celdas)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_matriz_segmentos.png', bbox_inches='tight')
plt.show()


Creamos el segmento de mercado combinado, uniendo valor RFM con perfil StepMix.


In [29]:
# Segmento combinado final: cada cliente queda asignado a un grupo RFM x StepMix
cust['segmento_mercado'] = (
    'RFM_' + cust['seg_rfm'].astype(str) +
    '_SOC_' + cust['seg_soc'].astype(str)
)

# Segmento combinado con nombres
cust['segmento_mercado_nombre'] = (
    cust['seg_rfm_nombre'] + ' + ' + cust['seg_soc_nombre']
)

print(f'Segmentos combinados distintos: {cust["segmento_mercado"].nunique()} ({K_RFM}x{K_SOC})')
print('\nDistribucion con nombres:')
print(cust['segmento_mercado_nombre'].value_counts())
cust[['customer_id', 'seg_rfm_nombre', 'seg_soc_nombre', 'segmento_mercado_nombre']].head(10)


Segmentos combinados distintos: 15 (3x5)

Distribucion con nombres:
segmento_mercado_nombre
Cliente Potencial + Smart Coffee             2276
Cliente Estrella + Smart Coffee              2062
Cliente Potencial + Classic n Quick          1559
Cliente Potencial + Practical Drive-Thru     1303
Cliente Estrella + Practical Drive-Thru      1180
Cliente Potencial + Suburban Pro             1165
Cliente Potencial + D-Frontier               1087
Cliente Estrella + D-Frontier                 910
Cliente Estrella + Suburban Pro               753
Cliente Espontaneo + Classic n Quick          725
Cliente Espontaneo + Smart Coffee             688
Cliente Espontaneo + Suburban Pro             384
Cliente Estrella + Classic n Quick            349
Cliente Espontaneo + D-Frontier               290
Cliente Espontaneo + Practical Drive-Thru     257
Name: count, dtype: int64


,customer_id,seg_rfm_nombre,seg_soc_nombre,segmento_mercado_nombre
0,CUST_00001,Cliente Estrella,Practical Drive-Thru,Cliente Estrella + Practical Drive-Thru
1,CUST_00002,Cliente Estrella,Smart Coffee,Cliente Estrella + Smart Coffee
2,CUST_00003,Cliente Potencial,Suburban Pro,Cliente Potencial + Suburban Pro
3,CUST_00004,Cliente Potencial,Practical Drive-Thru,Cliente Potencial + Practical Drive-Thru
4,CUST_00005,Cliente Potencial,Classic n Quick,Cliente Potencial + Classic n Quick
5,CUST_00007,Cliente Estrella,Suburban Pro,Cliente Estrella + Suburban Pro
6,CUST_00008,Cliente Potencial,Practical Drive-Thru,Cliente Potencial + Practical Drive-Thru
7,CUST_00009,Cliente Potencial,Smart Coffee,Cliente Potencial + Smart Coffee
8,CUST_00010,Cliente Potencial,Smart Coffee,Cliente Potencial + Smart Coffee
9,CUST_00011,Cliente Potencial,Practical Drive-Thru,Cliente Potencial + Practical Drive-Thru


---
## 8. Análisis de mercados meta

Construimos un resumen del atractivo comercial de los segmentos RFM.


In [30]:
# Perfil completo de segmentos RFM con info sociodemografica modal
perfil_completo = cust.groupby('seg_rfm').agg(
    n_clientes      = ('customer_id', 'count'),
    Recency_avg     = ('Recency',     'mean'),
    Frequency_avg   = ('Frequency',   'mean'),
    Monetary_avg    = ('Monetary',    'mean'),
    avg_spend_x_op  = ('avg_spend',   'mean'),
    pct_rewards     = ('pct_rewards', 'mean'),
    pct_order_ahead = ('pct_ahead',   'mean'),
    avg_satisf      = ('avg_satisf',  'mean'),
    region_top      = ('top_region',  lambda x: x.mode()[0]),
    location_top    = ('top_location',lambda x: x.mode()[0]),
    age_top         = ('top_age',     lambda x: x.mode()[0]),
    channel_top     = ('top_channel', lambda x: x.mode()[0]),
    gender_top      = ('top_gender',  lambda x: x.mode()[0]),
).round(2)
perfil_completo['pct_clientes'] = (perfil_completo['n_clientes'] / len(cust) * 100).round(1)

# Score de actividad normalizado (evita que Monetary domine por magnitud)
def norm_01(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)
rec_norm  = 1 - norm_01(perfil_completo['Recency_avg'])
freq_norm = norm_01(perfil_completo['Frequency_avg'])
mon_norm  = norm_01(perfil_completo['Monetary_avg'])
perfil_completo['score_actividad'] = ((rec_norm + freq_norm + mon_norm) / 3).round(4)
perfil_completo

,n_clientes,Recency_avg,Frequency_avg,Monetary_avg,avg_spend_x_op,pct_rewards,pct_order_ahead,avg_satisf,region_top,location_top,age_top,channel_top,gender_top,pct_clientes,score_actividad
seg_rfm,,,,,,,,,,,,,,,
0,2344,296.92,4.18,61.22,14.74,0.48,0.29,3.69,Midwest,Rural,25-34,Mobile App,Male,15.6,0.0000
1,5254,70.37,9.40,142.75,15.27,0.48,0.32,3.69,Midwest,Suburban,25-34,Mobile App,Female,35.1,1.0000
2,7390,76.03,5.52,80.27,14.63,0.47,0.29,3.69,Midwest,Rural,25-34,Mobile App,Female,49.3,0.4885


Ordenamos los segmentos RFM segun score de actividad para ver cuales son mas atractivos.


In [31]:
print('Ranking de segmentos RFM por score de actividad:')
perfil_completo[['n_clientes','pct_clientes','Recency_avg','Frequency_avg',
                 'Monetary_avg','score_actividad']]\
    .sort_values('score_actividad', ascending=False)

Ranking de segmentos RFM por score de actividad:


,n_clientes,pct_clientes,Recency_avg,Frequency_avg,Monetary_avg,score_actividad
seg_rfm,,,,,,
1,5254,35.1,70.37,9.40,142.75,1.0000
2,7390,49.3,76.03,5.52,80.27,0.4885
0,2344,15.6,296.92,4.18,61.22,0.0000


Graficamos frecuencia y gasto total usando los nombres de los segmentos RFM.


In [32]:
# Frecuencia vs Gasto total - dispersion por segmento RFM con nombres
fig, ax = plt.subplots(figsize=(9, 6))
colors_tab = plt.cm.tab10.colors
for seg in sorted(cust['seg_rfm'].unique()):
    sub = cust[cust['seg_rfm'] == seg]
    ax.scatter(sub['Frequency'], sub['Monetary'],
               c=[colors_tab[seg % 10]], alpha=0.35, s=9,
               label=nombres_rfm.get(seg, f'Seg. RFM {seg}'))
ax.set_xlabel('Frecuencia (numero de ordenes)')
ax.set_ylabel('Gasto total (USD)')
ax.set_title('Frecuencia vs Gasto total por segmento RFM')
ax.legend(markerscale=2.5, fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_freq_vs_monetary.png', bbox_inches='tight')
plt.show()


### 8.1 Mercados meta seleccionados por cuota de mercado

Se seleccionan tres mercados meta usando como primer criterio la mayor cuota de mercado dentro del cruce RFM + StepMix. Esta eleccion prioriza tamano, pero la recomendacion tambien considera actividad, gasto, canal dominante y tipo de local.


Seleccionamos tres mercados meta por mayor cuota de mercado y agregamos recomendaciones.


In [33]:
# Tabla de mercados meta: top 3 por cuota de mercado en el cruce RFM + StepMix
# La logica de recomendacion usa indices numericos, no nombres de segmentos.
# Asi, si se renombran los segmentos comerciales, las recomendaciones no se rompen.
def moda(x):
    return x.mode().iloc[0] if not x.mode().empty else np.nan

mercados_meta = (
    cust.groupby(['seg_rfm', 'seg_soc', 'seg_rfm_nombre', 'seg_soc_nombre', 'segmento_mercado_nombre'])
    .agg(
        n_clientes=('customer_id', 'count'),
        frequency_prom=('Frequency', 'mean'),
        monetary_prom=('Monetary', 'mean'),
        recency_prom=('Recency', 'mean'),
        satisfaccion_prom=('avg_satisf', 'mean'),
        rewards_prom=('pct_rewards', 'mean'),
        food_rate=('pct_food', 'mean'),
        order_ahead_rate=('pct_ahead', 'mean'),
        region_principal=('top_region', moda),
        tipo_local_principal=('top_location', moda),
        canal_principal=('top_channel', moda),
        edad_principal=('top_age', moda),
        genero_principal=('top_gender', moda),
    )
    .reset_index()
)
mercados_meta['pct_mercado'] = (mercados_meta['n_clientes'] / len(cust) * 100).round(2)

def recomendar(row):
    canal = row['canal_principal']
    local = row['tipo_local_principal']
    region = row['region_principal']

    if row['seg_rfm'] == 1:
        return (
            f'Mercado prioritario de alto valor. Reforzar {canal}, beneficios Rewards '
            f'y rapidez operativa en locales {local} de {region}.'
        )
    if row['seg_soc'] == 4:  # Mobile Coffee Fans
        return (
            f'Mercado grande y digital. Priorizar experiencia Mobile App, promociones Rewards '
            f'y order ahead en zonas {local} de {region}.'
        )
    if row['seg_soc'] == 3:  # Classic Speed Seniors
        return (
            f'Mercado amplio y de conveniencia. Mantener Drive-Thru simple, rapido y consistente '
            f'en tiendas {local} de {region}.'
        )
    return (
        f'Mercado con tamano relevante. Ajustar propuesta a canal {canal}, '
        f'tipo de local {local} y region {region}.'
    )

mercados_meta['recomendacion'] = mercados_meta.apply(recomendar, axis=1)
mercados_meta_top3 = mercados_meta.sort_values('pct_mercado', ascending=False).head(3).copy()

columnas_meta = [
    'segmento_mercado_nombre', 'n_clientes', 'pct_mercado',
    'frequency_prom', 'monetary_prom', 'recency_prom', 'satisfaccion_prom',
    'region_principal', 'tipo_local_principal', 'canal_principal',
    'rewards_prom', 'food_rate', 'order_ahead_rate', 'recomendacion'
]

print('Tres mercados meta seleccionados por mayor cuota de mercado:')
display(mercados_meta_top3[columnas_meta].round(3))


Tres mercados meta seleccionados por mayor cuota de mercado:


,segmento_mercado_nombre,n_clientes,pct_mercado,frequency_prom,monetary_prom,recency_prom,satisfaccion_prom,region_principal,tipo_local_principal,canal_principal,rewards_prom,food_rate,order_ahead_rate,recomendacion
14,Cliente Potencial + Smart Coffee,2276,15.19,5.345,87.412,77.280,3.796,Midwest,Rural,Mobile App,0.582,0.370,0.526,Mercado grande y digital. Priorizar experienci...
9,Cliente Estrella + Smart Coffee,2062,13.76,9.141,151.504,70.835,3.769,Midwest,Suburban,Mobile App,0.550,0.369,0.469,Mercado prioritario de alto valor. Reforzar Mo...
13,Cliente Potencial + Classic n Quick,1559,10.40,5.175,67.481,73.335,3.624,Midwest,Rural,Drive-Thru,0.385,0.274,0.000,Mercado amplio y de conveniencia. Mantener Dri...


Revisamos la presencia de segmentos RFM por region y tipo de local.


In [34]:
# Distribucion geografica y por tipo de local usando nombres RFM
df_merged = df.merge(cust[['customer_id', 'seg_rfm_nombre', 'seg_soc_nombre']], on='customer_id')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ct = df_merged.groupby(['region', 'seg_rfm_nombre']).size().unstack(fill_value=0)
ct = ct.reindex(columns=orden_rfm_nombres)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct.plot(kind='bar', ax=axes[0], colormap='tab10', edgecolor='white')
axes[0].set_title('Segmentos RFM por region')
axes[0].set_xlabel('Region'); axes[0].set_ylabel('% de transacciones')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Segmento RFM', bbox_to_anchor=(1, 1), fontsize=8)

ct2 = df_merged.groupby(['store_location_type', 'seg_rfm_nombre']).size().unstack(fill_value=0)
ct2 = ct2.reindex(columns=orden_rfm_nombres)
ct2_pct = ct2.div(ct2.sum(axis=1), axis=0) * 100
ct2_pct.plot(kind='bar', ax=axes[1], colormap='tab10', edgecolor='white')
axes[1].set_title('Segmentos RFM por tipo de local')
axes[1].set_xlabel('Tipo de local'); axes[1].set_ylabel('% de transacciones')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Segmento RFM', bbox_to_anchor=(1, 1), fontsize=8)

plt.suptitle('Distribucion geografica de segmentos RFM', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_region_segmentos.png', bbox_inches='tight')
plt.show()


---
## 9. Tabla resumen de segmentos

> **Nota:** La segmentacion sociodemografica/conductual fue recalculada con StepMix mixto. Los nombres comerciales ya fueron ajustados y validados a partir de los perfiles predominantes de cada segmento.

> Los segmentos relevantes para el cliente se determinan en base al score de actividad, el cruce RFM + socio y la interpretacion comercial del grupo.


Preparamos una tabla final para presentar los segmentos RFM de forma resumida.


In [35]:
resumen = perfil_completo[[
    'n_clientes', 'pct_clientes',
    'Recency_avg', 'Frequency_avg', 'Monetary_avg',
    'avg_spend_x_op', 'pct_rewards', 'pct_order_ahead', 'avg_satisf',
    'region_top', 'location_top', 'age_top', 'channel_top', 'gender_top',
    'score_actividad'
]].rename(columns={
    'n_clientes':       'N clientes',
    'pct_clientes':     '% mercado',
    'Recency_avg':      'Recency (dias)',
    'Frequency_avg':    'Freq. (ordenes)',
    'Monetary_avg':     'Gasto total (USD)',
    'avg_spend_x_op':   'Gasto x orden',
    'pct_rewards':      '% Rewards',
    'pct_order_ahead':  '% Order Ahead',
    'avg_satisf':       'Satisfaccion',
    'region_top':       'Region modal',
    'location_top':     'Local modal',
    'age_top':          'Edad modal',
    'channel_top':      'Canal modal',
    'gender_top':       'Genero modal',
    'score_actividad':  'Score actividad',
})

nombres_rfm = {0: 'Cliente Espontaneo', 1: 'Cliente Estrella', 2: 'Cliente Potencial'}
resumen.index = resumen.index.map(nombres_rfm)
resumen.round(2)

,N clientes,% mercado,Recency (dias),Freq. (ordenes),Gasto total (USD),Gasto x orden,% Rewards,% Order Ahead,Satisfaccion,Region modal,Local modal,Edad modal,Canal modal,Genero modal,Score actividad
seg_rfm,,,,,,,,,,,,,,,
Cliente Espontaneo,2344,15.6,296.92,4.18,61.22,14.74,0.48,0.29,3.69,Midwest,Rural,25-34,Mobile App,Male,0.00
Cliente Estrella,5254,35.1,70.37,9.40,142.75,15.27,0.48,0.32,3.69,Midwest,Suburban,25-34,Mobile App,Female,1.00
Cliente Potencial,7390,49.3,76.03,5.52,80.27,14.63,0.47,0.29,3.69,Midwest,Rural,25-34,Mobile App,Female,0.49


Exportamos los archivos finales de nuestra version StepMix.


In [36]:
# Exportar predicciones y variables originales (version v5)
cust.to_csv('clientes_segmentados_v5_stepmix.csv', index=False)
cruce.to_csv('matriz_segmentos_rfm_soc_v5_stepmix.csv')
resumen.to_csv('resumen_segmentos_rfm_v5_stepmix.csv')

# Exportar mercados meta seleccionados si la tabla ya fue calculada.
if 'mercados_meta_top3' in globals():
    mercados_meta_top3[columnas_meta].to_csv('mercados_meta_top3_v5_stepmix.csv', index=False)

print('Archivos exportados:')
print('  clientes_segmentados_v5_stepmix.csv - un cliente por fila con seg_rfm, seg_soc StepMix y segmento_mercado')
print('  matriz_segmentos_rfm_soc_v5_stepmix.csv - cruce RFM x Sociodemografico StepMix')
print('  resumen_segmentos_rfm_v5_stepmix.csv - perfil de cada segmento RFM')
if 'mercados_meta_top3' in globals():
    print('  mercados_meta_top3_v5_stepmix.csv - tres mercados meta seleccionados y recomendaciones')


Archivos exportados:
  clientes_segmentados_v5_stepmix.csv - un cliente por fila con seg_rfm, seg_soc StepMix y segmento_mercado
  matriz_segmentos_rfm_soc_v5_stepmix.csv - cruce RFM x Sociodemografico StepMix
  resumen_segmentos_rfm_v5_stepmix.csv - perfil de cada segmento RFM
  mercados_meta_top3_v5_stepmix.csv - tres mercados meta seleccionados y recomendaciones


---
## 10. Conclusiones

1. **Descripcion del contexto:** Se analizaron 100.000 transacciones de 14.988 clientes unicos de franquicias Starbucks en America. El objetivo es apoyar a un inversionista que evalua abrir nuevas franquicias, identificando clientes con alta actividad, valor comercial y perfiles accionables.

2. **Preparacion de datos:** La base original estaba a nivel de orden, por lo que se construyo una base final a nivel de cliente. Para cada cliente se resumieron variables de actividad, gasto, satisfaccion, uso de Rewards, compra de comida, pedido anticipado, canal principal, region principal, tipo de local principal y variables sociodemograficas.

3. **Segmentacion RFM:** La segmentacion RFM se mantuvo con K-Means usando `Recency`, `Frequency` y `Monetary`. Los segmentos quedan nombrados como **Cliente Espontaneo**, **Cliente Estrella** y **Cliente Potencial**, y estos nombres se usan tambien en los graficos para facilitar la lectura.

4. **Segmentacion sociodemografica/conductual con StepMix:** En esta version, la segmentacion sociodemografica/conductual usa **StepMix mixto**. Se selecciona **k = 5** porque entrega un equilibrio adecuado entre ajuste estadistico, entropia de clasificacion e interpretabilidad comercial. Los segmentos se nombraron a partir de sus rasgos predominantes: **Connected Professionals**, **Drive-Thru Traditionalists**, **Digital Frontier Users**, **Classic Speed Seniors** y **Mobile Coffee Fans**.

5. **Mercados meta seleccionados:** Se seleccionaron tres mercados meta usando como primer criterio la mayor cuota de mercado dentro del cruce RFM + StepMix:
   - **Cliente Potencial + Mobile Coffee Fans:** mercado grande, reciente y con orientacion digital. Recomendacion: reforzar Mobile App, Rewards y pedido anticipado para convertir potencial en mayor frecuencia.
   - **Cliente Estrella + Mobile Coffee Fans:** mercado de alto valor y alta actividad. Recomendacion: priorizar conveniencia digital, rapidez y beneficios de fidelizacion para retener clientes valiosos.
   - **Cliente Potencial + Classic Speed Seniors:** mercado amplio asociado a conveniencia y Drive-Thru. Recomendacion: mantener una experiencia simple, rapida y consistente en locales rurales.

6. **Implicancia para el inversionista:** La decision no debe basarse solo en tamano. La cuota de mercado ayuda a priorizar, pero debe cruzarse con gasto, frecuencia, recencia, satisfaccion y capacidad de ejecutar la propuesta en cada tipo de tienda. Los segmentos jovenes digitales sugieren reforzar canales moviles y Rewards, mientras que los segmentos Drive-Thru requieren foco en rapidez, accesibilidad y operacion eficiente.


In [39]:
# 11.1 Bebida favorita por segmento sociodemográfico (Por sí solo)
print("--- Bebida Favorita por Segmento Sociodemográfico ---")
# Usamos df_merged que ya contiene las columnas de transacciones y los segmentos asignados
fav_drink_soc = df_merged.groupby('seg_soc_nombre')['drink_category'].agg(lambda x: x.mode()[0]).reset_index()
fav_drink_soc.columns = ['Segmento Sociodemográfico', 'Bebida Favorita']
display(fav_drink_soc)

print("\n" + "="*60 + "\n")

# 11.2 Bebida favorita al cruzar Sociodemográfico x RFM
print("--- Bebida Favorita por Segmento Cruzado (Socio x RFM) ---")
# Calculamos la moda de la categoría de bebida para cada cruce
fav_drink_mixed = df_merged.groupby(['seg_soc_nombre', 'seg_rfm_nombre'])['drink_category'].agg(lambda x: x.mode()[0]).reset_index()

# Usamos pivot_table para mostrar la información en un formato de matriz fácil de leer
pivot_fav_drink = fav_drink_mixed.pivot(
    index='seg_soc_nombre', 
    columns='seg_rfm_nombre', 
    values='drink_category'
)
pivot_fav_drink.index.name = 'Segmento Sociodemográfico'
pivot_fav_drink.columns.name = 'Segmento RFM'
display(pivot_fav_drink)

# ====================================================================
# 11.3 Gráficos de la hora de la orden cruzado (Socio x RFM)
# ====================================================================
# Si por alguna razón 'order_hour' no está en df_merged, la recalculamos a partir de order_time:
if 'order_hour' not in df_merged.columns:
    df_merged['order_hour'] = pd.to_datetime(df_merged['order_time'], format='%H:%M').dt.hour

fig, axes = plt.subplots(2, 1, figsize=(14, 14))

# Gráfico 1: Boxplot de las horas de compra
# Permite ver la dispersión, la mediana de consumo y los valores atípicos de las horas de orden
sns.boxplot(
    data=df_merged, 
    x='seg_soc_nombre', 
    y='order_hour', 
    hue='seg_rfm_nombre', 
    palette='Set2', 
    ax=axes[0]
)
axes[0].set_title('Distribución de la Hora de Compra por Segmento Cruzado (Socio x RFM)', fontsize=14)
axes[0].set_xlabel('Segmento Sociodemográfico', fontsize=12)
axes[0].set_ylabel('Hora del Día (Order Hour)', fontsize=12)
axes[0].legend(title='Segmento RFM', bbox_to_anchor=(1.01, 1), loc='upper left')
axes[0].tick_params(axis='x', rotation=15)

# Gráfico 2: Heatmap de la Hora Promedio de Compra
# Da una visualización rápida de "calor" para identificar qué segmento compra más tarde o más temprano
heatmap_data = df_merged.groupby(['seg_soc_nombre', 'seg_rfm_nombre'])['order_hour'].mean().unstack()
sns.heatmap(
    heatmap_data, 
    annot=True, 
    fmt=".1f", 
    cmap="YlOrRd", 
    cbar_kws={'label': 'Hora Promedio de Compra'},
    ax=axes[1]
)
axes[1].set_title('Hora Promedio de Compra por Segmento Mixto', fontsize=14)
axes[1].set_xlabel('Segmento RFM', fontsize=12)
axes[1].set_ylabel('Segmento Sociodemográfico', fontsize=12)

plt.tight_layout()
plt.show()

--- Bebida Favorita por Segmento Sociodemográfico ---


,Segmento Sociodemográfico,Bebida Favorita
0,Classic n Quick,Brewed Coffee
1,D-Frontier,Espresso
2,Practical Drive-Thru,Tea
3,Smart Coffee,Refresher
4,Suburban Pro,Espresso




--- Bebida Favorita por Segmento Cruzado (Socio x RFM) ---


Segmento RFM,Cliente Espontaneo,Cliente Estrella,Cliente Potencial
Segmento Sociodemográfico,,,
Classic n Quick,Tea,Brewed Coffee,Refresher
D-Frontier,Espresso,Tea,Frappuccino
Practical Drive-Thru,Tea,Other,Tea
Smart Coffee,Refresher,Refresher,Tea
Suburban Pro,Refresher,Espresso,Espresso
